# learn-better — Google Colab setup

One-click setup for the [learn-better](https://github.com/dragosbo/learn-better) YouTube-to-study-material pipeline.

**How to use:** open this notebook in Colab (the *Open In Colab* badge in the repo README), then **Runtime → Run all**. Cells 1–2 install everything; cell 4 verifies; cell 5 does a real download + transcribe. The cells are idempotent — safe to re-run after a disconnect.

> Colab runs Linux, so the Windows `.bat` runners don't apply here — call the tools with `python code/<script>.py` (optionally a `config/*.json` argument). All outputs land under `data/` (see `lib/paths.py`).

## Cell 1 — System deps + clone the repo (run once per session)

In [ ]:
import os

# ffmpeg is a SYSTEM binary (not pip-installable) that yt-dlp/faster-whisper need.
!apt-get install -y ffmpeg -q

REPO_URL = "https://github.com/dragosbo/learn-better.git"
REPO_DIR = "/content/learn-better"

# Idempotent: only clone if it isn't already here.
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## Cell 2 — Python dependencies

Colab already ships its own Jupyter stack (`ipykernel`, `notebook`, …). Installing the repo's `ipykernel` pin on top of it triggers harmless-but-noisy `pip` "dependency conflict" warnings and can nag you to restart the runtime. `ipykernel` is only needed to run the notebooks in a *local* dev environment — on Colab it's redundant (Colab **is** the notebook host). So we install everything **except** `ipykernel` here to keep the output clean.

In [ ]:
# Install requirements WITHOUT ipykernel (Colab provides its own Jupyter stack;
# skipping it avoids the noisy 'dependency conflict' warnings + restart prompts).
with open("requirements.txt") as f:
    reqs = [ln for ln in f if ln.strip() and not ln.lstrip().lower().startswith("ipykernel")]
with open("/tmp/requirements.colab.txt", "w") as f:
    f.writelines(reqs)

!pip install -q -r /tmp/requirements.colab.txt
print("\n✅ Dependencies installed (ipykernel skipped — Colab supplies it).")

## Cell 3 (OPTIONAL) — Persist outputs to Google Drive

**You can skip this cell.** The pipeline works fully without it — the only difference is that everything under `data/` lives in the temporary Colab session and is **lost when the runtime disconnects**. Mount Drive if you want your audio / transcripts / summaries to survive between sessions.

If the mount fails (`ValueError: mount failed`), it's almost always the Google auth popup being dismissed, timing out, or blocked by third-party cookies — **not** a problem with this project. The cell below catches that so it won't stop **Run all**; just re-run it and complete the popup (grant all permissions, same Google account). Files still work without Drive — see "Accessing your files" below.

In [ ]:
import os

DRIVE_OUT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT = "/content/drive/MyDrive/learn-better-output"
    # Subdirs mirror lib/paths.py (all outputs live under data/).
    for sub in ["audio", "audio_reencoded", "transcripts", "generated_transcripts",
                "summaries", "tts_output", "wordclouds", "whisper-model-cache"]:
        os.makedirs(f"{DRIVE_OUT}/{sub}", exist_ok=True)
    print(f"✅ Drive mounted. Persist outputs by copying data/ into: {DRIVE_OUT}")
except Exception as e:
    print(f"⚠️  Drive NOT mounted ({type(e).__name__}: {e}).")
    print("   This is optional — continuing without Drive. Outputs stay under")
    print("   /content/learn-better/data and vanish when the session ends.")
    print("   To retry: re-run this cell and complete the Google popup fully.")

## Cell 4 — Verify the environment

In [ ]:
# GPU check (optional): a T4 makes Whisper 5-10x faster. 'command not found' just
# means CPU-only (fine, slower on large models). To get a GPU:
# Runtime -> Change runtime type -> T4 GPU, then reconnect. faster-whisper uses CUDA automatically.
!nvidia-smi | head -3 || echo 'No GPU (CPU is fine; large Whisper models are slower).'

!ffmpeg -version | head -1
import yt_dlp, faster_whisper, pandas
print('✅ All imports OK — environment ready.')

## Cell 5 — Example run (download → transcribe)

The tools are config-driven and follow a pipeline: **first download** audio + transcripts for a source, **then** transcribe/summarize what you downloaded. A fresh clone has an empty `data/audio/`, so we download first — otherwise the transcribe step correctly reports *"No audio files matched"* (nothing to do yet).

`read_channel.py` reads its source from the config block at the top of the file (`PLAYLIST_ID` / `CHANNEL` / `SEARCH` / `LIMIT`). The cell below points it at a search with `sed`, then runs the download + a Whisper transcription. Edit the search term to whatever you want.

In [ ]:
# 1) Point read_channel.py at a search (or set PLAYLIST_ID / CHANNEL instead), keep it small.
!sed -i 's|^SEARCH *=.*|SEARCH = "git tutorial for beginners"|' code/read_channel.py
!sed -i 's|^LIMIT *=.*|LIMIT = 1|' code/read_channel.py

# 2) Download audio + transcripts -> data/audio/ , data/transcripts/
!python code/read_channel.py

# 3) Transcribe the downloaded audio with Whisper -> data/generated_transcripts/
#    (uses config/config_transcribe.json; edit it to target specific files)
!python code/transcribe_audio.py config/config_transcribe.json

## Accessing your files

All outputs land under `data/` inside the repo: `data/audio/`, `data/transcripts/`, `data/generated_transcripts/`, `data/summaries/`, `data/tts_output/`, `data/wordclouds/`.

Three ways to get at them:
1. **File browser (easiest):** click the 📁 folder icon in Colab's left sidebar → expand `learn-better/data/` → right-click a file → **Download**.
2. **List them in a cell:** run the cell below.
3. **Download one file to your computer:** `from google.colab import files; files.download('<path>')`.

> **Persistence:** Colab wipes `/content/` when the session ends. To keep files, either download them, or mount Drive (Cell 3) and copy `data/` there (the last cell does this automatically if Drive is mounted).

In [ ]:
# List everything produced under data/
!ls -R /content/learn-better/data 2>/dev/null || echo 'No data/ yet — run the example (Cell 5) first.'

# If Drive was mounted (Cell 3), copy all outputs there so they survive the session.
import os, shutil, pathlib
if 'DRIVE_OUT' in globals() and DRIVE_OUT:
    data = pathlib.Path('/content/learn-better/data')
    if data.is_dir():
        for src in data.rglob('*'):
            if src.is_file():
                dest = pathlib.Path(DRIVE_OUT) / src.relative_to(data)
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(src, dest)
        print(f'✅ Copied data/ -> {DRIVE_OUT}')
    else:
        print('No data/ to copy yet.')
else:
    print('Drive not mounted — download files from the 📁 sidebar before disconnecting.')